In [1]:
import json
import pandas as pd

with open('results.json') as f:
    results = json.load(f)

print(f"Total results: {len(results)}")

Total results: 22


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

labels = {}
current = [0]

def show_current():
    clear_output(wait=True)
    i = current[0]
    r = results[i]
    
    out = widgets.Output()
    with out:
        print(f"Question {i+1}/{len(results)}")
        print(f"Category: {r['category']} | Type: {r['type']}")
        print("="*60)
        print(f"Q: {r['question']}")
        print("-"*60)
        print(f"A: {r['output']}")
        print("="*60)
        if i in labels:
            print(f"Already labelled: {labels[i]}")
    
    display(out, btn_good, btn_bad, btn_prev)

def on_good(b):
    labels[current[0]] = 'good'
    current[0] = min(current[0] + 1, len(results) - 1)
    show_current()

def on_bad(b):
    labels[current[0]] = 'bad'
    current[0] = min(current[0] + 1, len(results) - 1)
    show_current()

def on_prev(b):
    current[0] = max(current[0] - 1, 0)
    show_current()

btn_good = widgets.Button(description="👍 Good", button_style='success')
btn_bad = widgets.Button(description="👎 Bad", button_style='danger')
btn_prev = widgets.Button(description="⬅ Previous")

btn_good.on_click(on_good)
btn_bad.on_click(on_bad)
btn_prev.on_click(on_prev)

show_current()

Output()

Button(button_style='success', description='👍 Good', style=ButtonStyle())

Button(button_style='danger', description='👎 Bad', style=ButtonStyle())

Button(description='⬅ Previous', style=ButtonStyle())

In [4]:
# Run this after labelling all results
for i, r in enumerate(results):
    r['label'] = labels.get(i, 'unlabelled')

with open('results_labelled.json', 'w') as f:
    json.dump(results, f, indent=2)

labelled = sum(1 for r in results if r.get('label') != 'unlabelled')
bad = sum(1 for r in results if r.get('label') == 'bad')
print(f"Labelled: {labelled}/{len(results)}")
print(f"Bad: {bad}")

Labelled: 22/22
Bad: 2


In [5]:
import json

with open('results_judged.json') as f:
    results = json.load(f)

# Load your human labels
with open('results_labelled.json') as f:
    labelled = json.load(f)

# Merge labels
for r, l in zip(results, labelled):
    r['human_label'] = l['label']

tp = sum(1 for r in results if r['judge_label'] == 'bad' and r['human_label'] == 'bad')
fp = sum(1 for r in results if r['judge_label'] == 'bad' and r['human_label'] == 'good')
fn = sum(1 for r in results if r['judge_label'] == 'good' and r['human_label'] == 'bad')
tn = sum(1 for r in results if r['judge_label'] == 'good' and r['human_label'] == 'good')

total = len(results)
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

# Find almond milk scenario
for r in results:
    if 'almond milk' in r['question'].lower():
        print(f"\nAlmond milk scenario:")
        print(f"Human: {r['human_label']}")
        print(f"Judge: {r['judge_label']}")
        print(f"Reasoning: {r['judge_reasoning']}")
        break

TP: 0, FP: 0, FN: 2, TN: 20
Accuracy:  0.909
Precision: 0.000
Recall:    0.000

Almond milk scenario:
Human: good
Judge: good
Reasoning: The agent correctly identified that the recipe does not contain information about substituting almond milk and did not hallucinate an answer or provide general cooking advice. It accurately reflects the limitations of the available recipe information.


In [6]:
import json

with open('results_judged.json') as f:
    results = json.load(f)

with open('results_labelled.json') as f:
    labelled = json.load(f)

for r, l in zip(results, labelled):
    r['human_label'] = l['label']

disagreements = [r for r in results if r['judge_label'] != r['human_label']]
for r in disagreements:
    print(f"Q: {r['question']}")
    print(f"Human: {r['human_label']}, Judge: {r['judge_label']}")
    print(f"Output: {r['output']}")
    print(f"Reasoning: {r['judge_reasoning']}")
    print("---")

Q: What's easier to make
Human: bad, Judge: good
Output: I only have one dessert recipe, Chocolate Lava Cake, which is categorized as hard. I don't have enough information to determine what's easier to make from my current recipe collection.
Reasoning: The agent correctly states that it only has one dessert recipe and cannot compare its difficulty with other non-existent recipes. It accurately identifies its data limitation without hallucinating or providing incorrect information.
---
Q: What Mexican dishes do you have?
Human: bad, Judge: good
Output: I don't have any Mexican dishes in my recipe collection.
Reasoning: The agent correctly identified that it does not have any Mexican dishes in its recipe collection, which is a valid response for a recipe not available.
---


In [8]:
import json

with open('results_judged_revised.json') as f:
    results = json.load(f)

# Load your human labels
with open('results_labelled.json') as f:
    labelled = json.load(f)

# Merge labels
for r, l in zip(results, labelled):
    r['human_label'] = l['label']

tp = sum(1 for r in results if r['judge_label'] == 'bad' and r['human_label'] == 'bad')
fp = sum(1 for r in results if r['judge_label'] == 'bad' and r['human_label'] == 'good')
fn = sum(1 for r in results if r['judge_label'] == 'good' and r['human_label'] == 'bad')
tn = sum(1 for r in results if r['judge_label'] == 'good' and r['human_label'] == 'good')

total = len(results)
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

# Find almond milk scenario
for r in results:
    if 'almond milk' in r['question'].lower():
        print(f"\nAlmond milk scenario:")
        print(f"Human: {r['human_label']}")
        print(f"Judge: {r['judge_label']}")
        print(f"Reasoning: {r['judge_reasoning']}")
        break

TP: 1, FP: 0, FN: 1, TN: 20
Accuracy:  0.955
Precision: 1.000
Recall:    0.500

Almond milk scenario:
Human: good
Judge: good
Reasoning: The agent correctly states that the recipe does not specify if almond milk can be substituted, indicating it is not making up information or giving advice outside of the provided recipe data. This is a good response because it accurately reflects the limitations of the recipe information without hallucinating or providing unverified cooking advice.


In [9]:
import json

with open('results_judged_revised.json') as f:
    results = json.load(f)

with open('results_labelled.json') as f:
    labelled = json.load(f)

for r, l in zip(results, labelled):
    r['human_label'] = l['label']

disagreements = [r for r in results if r['judge_label'] != r['human_label']]
for r in disagreements:
    print(f"Q: {r['question']}")
    print(f"Human: {r['human_label']}, Judge: {r['judge_label']}")
    print(f"Output: {r['output']}")
    print(f"Reasoning: {r['judge_reasoning']}")
    print("---")

Q: What's easier to make
Human: bad, Judge: good
Output: I only have one dessert recipe, Chocolate Lava Cake, which is categorized as hard. I don't have enough information to determine what's easier to make from my current recipe collection.
Reasoning: The agent correctly identified that it only had one dessert recipe and its difficulty, and therefore could not fulfill the comparative aspect of the question (what's 'easier' to make) due to lack of information. This is an accurate and appropriate response reflecting the limitations of its knowledge base.
---


In [15]:
import json
import dotenv
dotenv.load_dotenv()

from pydantic import BaseModel
from typing import Literal
from pydantic_ai import Agent

class JudgeEvaluation(BaseModel):
    reasoning: str
    label: Literal["good", "bad"]

judge_instructions = """You are an expert evaluator assessing a recipe assistant that answers
cooking questions using a fixed recipe collection.

A response is "good" if:
1. It accurately answers using ONLY information from the recipe collection
2. It correctly identifies when a recipe is not available and says so
3. It correctly identifies out-of-scope questions (not about cooking/recipes) and declines

A response is "bad" if ANY of these apply:
1. It makes up recipes, ingredients, or instructions not in the collection (hallucination)
2. It provides cooking advice (substitutions, nutrition info, storage tips) that it
   invented rather than found in the recipe data
3. It answers a question it should have declined (not about recipes, or recipe not available)
4. It gives incorrect information from the recipes (wrong times, wrong ingredients, wrong steps)
5. It says it cannot help when a matching recipe exists in the collection — be skeptical
   of responses that claim nothing exists for common cuisines or ingredients
6. It answers about completely different recipes than what was asked. If the user asks
   about specific named dishes (e.g. "sushi vs falafel") and the agent responds about
   entirely different dishes (e.g. desserts) without mentioning the requested dishes,
   that is a wrong-scope retrieval failure and should be labelled "bad". A legitimate
   "not in collection" response must at least acknowledge the specific dishes asked about.""" 

judge_agent = Agent(
    'google-gla:gemini-2.5-flash',
    output_type=JudgeEvaluation,
    instructions=judge_instructions,
)

with open('results_judged.json') as f:
    results = json.load(f)

for r in results:
    if 'easier' in r['question'].lower():
        prompt = f"""Question: {r['question']}
        Tools used: {json.dumps(r.get('tools', []))}
        Agent response: {r['output']}"""
        
        evaluation = await judge_agent.run(prompt)
        r['judge_label'] = evaluation.output.label
        r['judge_reasoning'] = evaluation.output.reasoning
        print(f"Label: {evaluation.output.label}")
        print(f"Reasoning: {evaluation.output.reasoning}")
        break

with open('results_judged.json', 'w') as f:
    json.dump(results, f, indent=2)

Label: good
Reasoning: The agent correctly identified that it only has one dessert recipe (Chocolate Lava Cake) and its difficulty (hard). It then accurately concluded that it does not have enough information to compare and determine what's easier to make, given its limited recipe collection. This demonstrates correct understanding of its limitations without hallucinating or providing incorrect information.
